In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
DATASET_DIR = PROJECT_DIR / "datasets"

YOLO_RAW = DATASET_DIR / "raw" / "yolo_sources"
STATE_RAW = DATASET_DIR / "raw" / "state_sources"

for folder in [YOLO_RAW, STATE_RAW]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("YOLO dataset:", YOLO_RAW)
print("State dataset:", STATE_RAW)

In [ ]:
%pip install -U kagglehub requests tqdm pyyaml pillow

In [ ]:
%pip install "Pillow>=11.0,<12.0"

In [ ]:
import sys
import PIL
import kagglehub
import tqdm

print("Python:", sys.version)
print("Pillow:", PIL.__version__)
print("KaggleHub:", kagglehub.__version__)
print("TQDM:", tqdm.__version__)
print("Các thư viện đã import thành công!")

In [ ]:
%pip check

In [ ]:
%pip check

In [ ]:
from pathlib import Path
import kagglehub

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

STATE_RAW = (
    PROJECT_DIR
    / "datasets"
    / "raw"
    / "state_sources"
)

STATE_RAW.mkdir(parents=True, exist_ok=True)

path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification",
    output_dir=str(STATE_RAW / "Fresh_Rotten_Fruits")
)

print("Dataset lưu tại:", path)

In [ ]:
%pip install "Pillow>=11.0,<12.0"

In [ ]:
%pip check

In [ ]:
from pathlib import Path

DATASET_PATH = Path(
    r"C:\Users\ASUS\SIC-Food\datasets\raw"
    r"\state_sources\Fresh_Rotten_Fruits"
)

print("Dataset tồn tại:", DATASET_PATH.exists())

for item in DATASET_PATH.iterdir():
    print("📁" if item.is_dir() else "📄", item.name)

In [ ]:
from collections import Counter
import pandas as pd

extensions = {".jpg", ".jpeg", ".png", ".webp"}

counts = Counter()

for path in DATASET_PATH.rglob("*"):
    if path.is_file() and path.suffix.lower() in extensions:
        relative = path.parent.relative_to(DATASET_PATH)
        counts[str(relative)] += 1

df = pd.DataFrame(
    counts.items(),
    columns=["Thư mục", "Số lượng ảnh"]
).sort_values("Thư mục")

display(df)

print("TỔNG SỐ ẢNH:", df["Số lượng ảnh"].sum())

In [ ]:
from PIL import Image
from IPython.display import display
import random

image_files = [
    p for p in DATASET_PATH.rglob("*")
    if p.is_file() and p.suffix.lower()
    in {".jpg", ".jpeg", ".png", ".webp"}
]

print("Tổng ảnh tìm được:", len(image_files))

sample_images = random.sample(
    image_files,
    min(9, len(image_files))
)

for image_path in sample_images:
    print("Nhãn thư mục:", image_path.parent.name)
    print("Tên ảnh:", image_path.name)

    try:
        with Image.open(image_path) as image:
            preview = image.copy()
            preview.thumbnail((300, 300))
            display(preview)
    except Exception as e:
        print("Ảnh lỗi:", e)

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib

hash_groups = defaultdict(list)

for path in image_files:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    hash_groups[h.hexdigest()].append(path)

duplicates = {
    h: paths
    for h, paths in hash_groups.items()
    if len(paths) > 1
}

duplicate_extra = sum(len(paths) - 1 for paths in duplicates.values())

print("Tổng file ảnh:", len(image_files))
print("Số ảnh trùng hoàn toàn (bản dư):", duplicate_extra)
print("Số nhóm ảnh trùng:", len(duplicates))

for paths in list(duplicates.values())[:5]:
    print("\nNhóm trùng:")
    for path in paths:
        print(path.relative_to(DATASET_PATH))

In [ ]:
from collections import defaultdict

split_hashes = defaultdict(set)

for h, paths in hash_groups.items():
    splits = set()

    for path in paths:
        relative_parts = [
            part.lower()
            for part in path.relative_to(DATASET_PATH).parts
        ]

        if "train" in relative_parts:
            splits.add("train")

        if "test" in relative_parts:
            splits.add("test")

    if len(splits) > 1:
        split_hashes[h] = splits

print("Số nhóm ảnh trùng giữa Train và Test:", len(split_hashes))

In [ ]:
from pathlib import Path

root = DATASET_PATH

candidate_1 = root / "dataset"
candidate_2 = root / "dataset" / "dataset"

for candidate in [candidate_1, candidate_2]:
    print("\nĐường dẫn:", candidate)
    print("Tồn tại:", candidate.exists())

    if candidate.exists():
        for name in ["train", "test"]:
            folder = candidate / name
            images = [
                p for p in folder.rglob("*")
                if p.is_file()
                and p.suffix.lower() in {
                    ".jpg", ".jpeg", ".png", ".webp"
                }
            ] if folder.exists() else []

            print(f"{name}: {len(images)} ảnh")

In [ ]:
SOURCE_DATASET = DATASET_PATH / "dataset"

print("Dataset chuẩn:", SOURCE_DATASET)
print("Train:", (SOURCE_DATASET / "train").exists())
print("Test:", (SOURCE_DATASET / "test").exists())

In [ ]:
import pandas as pd

records = []

for split in ["train", "test"]:
    split_dir = SOURCE_DATASET / split

    if not split_dir.exists():
        continue

    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        images = [
            p for p in class_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower() in {
                ".jpg", ".jpeg", ".png", ".webp"
            }
        ]

        records.append({
            "Split": split,
            "Class": class_dir.name,
            "Images": len(images)
        })

df_summary = pd.DataFrame(records)

display(
    df_summary.pivot_table(
        index="Class",
        columns="Split",
        values="Images",
        aggfunc="sum",
        fill_value=0
    )
)

print("Tổng ảnh:", df_summary["Images"].sum())

In [ ]:
from collections import Counter

extensions = {".jpg", ".jpeg", ".png", ".webp"}

for class_dir in sorted((SOURCE_DATASET / "train").iterdir()):
    if not class_dir.is_dir():
        continue

    files = sorted(
        p for p in class_dir.iterdir()
        if p.suffix.lower() in extensions
    )

    print(f"\n===== {class_dir.name} =====")
    print("Tổng ảnh:", len(files))

    for image in files[:12]:
        print(image.name)

In [ ]:
from collections import Counter

prefixes = Counter()

for path in (SOURCE_DATASET / "train").rglob("*"):
    if path.is_file() and path.suffix.lower() in extensions:
        name = path.name.lower()

        if name.startswith("rotated_by_"):
            prefixes["rotated_by"] += 1
        elif name.startswith("horizontal_flip"):
            prefixes["horizontal_flip"] += 1
        elif name.startswith("vertical_flip"):
            prefixes["vertical_flip"] += 1
        elif name.startswith("salt_and_pepper"):
            prefixes["salt_and_pepper"] += 1
        else:
            prefixes["original_or_other"] += 1

for prefix, count in prefixes.most_common():
    print(f"{prefix}: {count}")

In [ ]:
from collections import Counter

other_files = []

for path in (SOURCE_DATASET / "train").rglob("*"):
    if path.is_file() and path.suffix.lower() in extensions:
        name = path.name.lower()

        if not (
            name.startswith("rotated_by_")
            or name.startswith("vertical_flip")
            or name.startswith("horizontal_flip")
            or name.startswith("salt_and_pepper")
        ):
            other_files.append(path.name)

print("Số ảnh thuộc nhóm khác:", len(other_files))

for name in sorted(set(other_files))[:30]:
    print(name)

In [ ]:
import re
from collections import defaultdict

def get_original_name(filename):
    name = filename

    patterns = [
        r"^rotated_by_[+-]?\d+(?:\.\d+)?_",
        r"^vertical_flip_?",
        r"^horizontal_flip_?",
        r"^salt_and_pepper_?",
    ]

    changed = True

    while changed:
        changed = False

        for pattern in patterns:
            new_name = re.sub(
                pattern, "", name,
                flags=re.IGNORECASE
            )

            if new_name != name:
                name = new_name
                changed = True

    return name.lower()

groups = defaultdict(list)

for path in (SOURCE_DATASET / "train").rglob("*"):
    if path.is_file() and path.suffix.lower() in extensions:

        original = get_original_name(path.name)

        key = (
            path.parent.name.lower(),
            original
        )

        groups[key].append(path)

print("Tổng ảnh Train:", sum(len(v) for v in groups.values()))
print("Số nhóm tên ảnh gốc:", len(groups))
print("Số nhóm có nhiều ảnh:", sum(len(v) > 1 for v in groups.values()))

In [ ]:
def get_group_keys(split):
    result = set()

    folder = SOURCE_DATASET / split

    for path in folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in extensions:
            key = (
                path.parent.name.lower(),
                get_original_name(path.name)
            )
            result.add(key)

    return result

train_keys = get_group_keys("train")
test_keys = get_group_keys("test")

overlap = train_keys.intersection(test_keys)

print("Số nhóm Train:", len(train_keys))
print("Số nhóm Test:", len(test_keys))
print("Số nhóm nghi trùng Train/Test:", len(overlap))

for item in sorted(overlap)[:20]:
    print(item)

In [ ]:
import pandas as pd

rows = []

for (class_name, original_name), paths in groups.items():
    rows.append({
        "class": class_name,
        "original_name": original_name,
        "number_of_images": len(paths),
        "sample_path": str(paths[0]),
    })

df_groups = pd.DataFrame(rows)

report_dir = PROJECT_DIR / "datasets" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / "augmentation_groups.csv"

df_groups.to_csv(
    report_path,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu báo cáo:", report_path)

display(
    df_groups.groupby("class").agg(
        groups=("original_name", "count"),
        images=("number_of_images", "sum")
    )
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

print("Đường dẫn dự án:", PROJECT_DIR)
print("Tồn tại:", PROJECT_DIR.exists())

In [ ]:
print("Tổng ảnh Train:", sum(len(v) for v in groups.values()))
print("Số nhóm ảnh gốc:", len(groups))
print("Số nhóm có nhiều ảnh:", sum(len(v) > 1 for v in groups.values()))

print("\nCác nhóm mẫu:")

for (class_name, original_name), paths in list(groups.items())[:5]:
    print("\nLớp:", class_name)
    print("Tên gốc:", original_name)
    print("Số ảnh:", len(paths))

    for p in paths[:5]:
        print(" -", p.name)

In [ ]:
from collections import defaultdict
import pandas as pd

records = []

for split in ["train", "test"]:
    split_dir = SOURCE_DATASET / split

    for image in split_dir.rglob("*"):
        if not image.is_file():
            continue

        if image.suffix.lower() not in extensions:
            continue

        records.append({
            "split": split,
            "class": image.parent.name,
            "original_name": get_original_name(image.name),
            "path": str(image)
        })

df_manifest = pd.DataFrame(records)

group_cols = ["class", "original_name"]

leakage = (
    df_manifest.groupby(group_cols)["split"]
    .nunique()
    .reset_index(name="split_count")
)

leakage = leakage[leakage["split_count"] > 1]

print("Tổng ảnh trong manifest:", len(df_manifest))
print("Số nhóm nghi trùng giữa Train/Test:", len(leakage))

display(leakage.head(20))

In [ ]:
report_dir = PROJECT_DIR / "datasets" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

manifest_path = report_dir / "state_dataset_manifest.csv"
leakage_path = report_dir / "potential_leakage.csv"

df_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
leakage.to_csv(leakage_path, index=False, encoding="utf-8-sig")

print("Đã lưu manifest:", manifest_path)
print("Đã lưu báo cáo nghi trùng:", leakage_path)

In [ ]:
from pathlib import Path
from collections import defaultdict
import re
import random
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

DATASET_PATH = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "Fresh_Rotten_Fruits"
)

SOURCE_DATASET = DATASET_PATH / "dataset"
OUTPUT_DATASET = PROJECT_DIR / "datasets" / "efficientnet"
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_MAP = {
    "freshapples": "apple__normal_appearance",
    "rottenapples": "apple__visible_defect",
    "freshbanana": "banana__normal_appearance",
    "rottenbanana": "banana__visible_defect",
    "freshoranges": "orange__normal_appearance",
    "rottenoranges": "orange__visible_defect",
}

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

# Loại các tiền tố augmentation được nhận diện
PATTERNS = [
    r"^rotated_by_[+-]?\d+(?:\.\d+)?_",
    r"^vertical_flip_?",
    r"^horizontal_flip_?",
    r"^salt_and_pepper_?",
]

def get_original_name(filename):
    name = filename.lower()
    changed = True

    while changed:
        changed = False

        for pattern in PATTERNS:
            new_name = re.sub(pattern, "", name)

            if new_name != name:
                name = new_name
                changed = True

    return name

# Đọc dữ liệu từ một bản sao duy nhất
records = []

for source_split in ["train", "test"]:
    split_dir = SOURCE_DATASET / source_split

    if not split_dir.is_dir():
        raise FileNotFoundError(split_dir)

    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        if class_dir.name not in CLASS_MAP:
            print("Bỏ qua lớp chưa xác minh:", class_dir.name)
            continue

        for path in class_dir.rglob("*"):
            if not path.is_file():
                continue

            if path.suffix.lower() not in EXTENSIONS:
                continue

            original_name = get_original_name(path.name)

            records.append({
                "path": str(path),
                "source_split": source_split,
                "original_class": class_dir.name,
                "class": CLASS_MAP[class_dir.name],
                "original_name": original_name,
                "group_key": CLASS_MAP[class_dir.name] + "::" + original_name
            })

df_source = pd.DataFrame(records)

if df_source.empty:
    raise ValueError("Không tìm thấy dữ liệu đầu vào")

# Phát hiện nhóm ảnh xuất hiện ở nhiều split
split_counts = (
    df_source.groupby("group_key")["source_split"]
    .nunique()
)

leaked_keys = set(
    split_counts[split_counts > 1].index
)

print("Tổng ảnh nguồn:", len(df_source))
print("Số nhóm nghi rò rỉ Train/Test:", len(leaked_keys))

# Cách ly toàn bộ ảnh thuộc nhóm nghi rò rỉ
df_source["quarantined"] = (
    df_source["group_key"].isin(leaked_keys)
)

safe_data = df_source[~df_source["quarantined"]].copy()

# Giữ Test gốc không xung đột
df_test = safe_data[
    safe_data["source_split"] == "test"
].copy()

df_train_source = safe_data[
    safe_data["source_split"] == "train"
].copy()

# Chia Validation theo nhóm ảnh nguồn,
# không chia ngẫu nhiên từng ảnh
random_generator = random.Random(42)
validation_keys = set()

for class_name, class_data in df_train_source.groupby("class"):
    group_keys = sorted(class_data["group_key"].unique())

    if len(group_keys) < 3:
        raise ValueError(
            f"Lớp {class_name} không đủ nhóm độc lập"
        )

    random_generator.shuffle(group_keys)

    # Khoảng 18,75% nhóm Train gốc cho Validation
    n_val = max(1, round(len(group_keys) * 0.1875))
    n_val = min(n_val, len(group_keys) - 1)

    validation_keys.update(group_keys[:n_val])

df_train_source["split"] = df_train_source["group_key"].apply(
    lambda key: "val" if key in validation_keys else "train"
)

df_test["split"] = "test"

df_ready = pd.concat(
    [df_train_source, df_test],
    ignore_index=True
)

# Đảm bảo một nhóm nguồn chỉ thuộc một split
assert (
    df_ready.groupby("group_key")["split"].nunique().max()
    == 1
)

# Đếm theo số ảnh thực tế
summary = pd.crosstab(
    df_ready["class"],
    df_ready["split"]
)

display(summary)

print("\nTổng ảnh được chọn:", len(df_ready))
print("Ảnh cách ly:", int(df_source["quarantined"].sum()))
print("Tổng nhóm Train/Val/Test:", df_ready["group_key"].nunique())

# Lưu kế hoạch
df_ready.to_csv(
    REPORT_DIR / "efficientnet_split_plan.csv",
    index=False,
    encoding="utf-8-sig"
)

df_source[df_source["quarantined"]].to_csv(
    REPORT_DIR / "quarantined_images.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã tạo kế hoạch, chưa sao chép ảnh.")

In [ ]:
import shutil
import hashlib
from PIL import Image
from tqdm.auto import tqdm

# Không ghi đè dataset đã xuất trước đó
if OUTPUT_DATASET.exists():
    existing_images = [
        p for p in OUTPUT_DATASET.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    ]

    if existing_images:
        raise RuntimeError(
            "Thư mục EfficientNet đã có ảnh. "
            "Hãy kiểm tra dữ liệu cũ trước khi xuất lại."
        )

# Kiểm tra dung lượng ổ đĩa
required_bytes = sum(
    Path(p).stat().st_size
    for p in df_ready["path"]
)

free_bytes = shutil.disk_usage(OUTPUT_DATASET.parent).free

print("Dung lượng cần:", round(required_bytes / 1024**3, 2), "GB")
print("Dung lượng còn trống:", round(free_bytes / 1024**3, 2), "GB")

if free_bytes < required_bytes * 1.15:
    raise RuntimeError("Không đủ dung lượng để sao chép dataset")

export_records = []
failed_images = []

for row in tqdm(
    df_ready.itertuples(index=False),
    total=len(df_ready),
    desc="Export EfficientNet"
):
    source = Path(row.path)

    target_dir = OUTPUT_DATASET / row.split / row._asdict()["class"]
    target_dir.mkdir(parents=True, exist_ok=True)

    # Tạo tên duy nhất theo đường dẫn gốc
    relative = str(source.relative_to(SOURCE_DATASET))
    unique_id = hashlib.sha1(relative.encode("utf-8")).hexdigest()[:16]

    target = target_dir / f"{unique_id}{source.suffix.lower()}"

    try:
        # Kiểm tra file ảnh trước khi sao chép
        with Image.open(source) as img:
            img.verify()

        shutil.copy2(source, target)

        export_records.append({
            "source_path": str(source),
            "target_path": str(target),
            "split": row.split,
            "class": row._asdict()["class"],
            "original_class": row.original_class,
            "group_key": row.group_key
        })

    except Exception as error:
        failed_images.append({
            "source_path": str(source),
            "error": str(error)
        })

export_df = pd.DataFrame(export_records)

export_df.to_csv(
    REPORT_DIR / "efficientnet_export_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    failed_images,
    columns=["source_path", "error"]
).to_csv(
    REPORT_DIR / "efficientnet_failed_images.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nHOÀN THÀNH XUẤT DATASET")
print("Ảnh sao chép thành công:", len(export_records))
print("Ảnh lỗi:", len(failed_images))
print("Đường dẫn:", OUTPUT_DATASET)

In [ ]:
import shutil
import hashlib
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

# Kiểm tra dữ liệu đã chuẩn bị
assert not df_ready.empty, "Dataset đang trống"

required_columns = {
    "path", "split", "class",
    "original_class", "group_key"
}

assert required_columns.issubset(df_ready.columns)

# Không ghi đè ảnh đã xuất trước đó
if OUTPUT_DATASET.exists():
    existing_images = [
        p for p in OUTPUT_DATASET.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    ]

    if existing_images:
        raise RuntimeError(
            "Thư mục EfficientNet đã có ảnh. "
            "Hãy kiểm tra trước khi chạy lại."
        )

# Kiểm tra dung lượng ổ đĩa
required_bytes = sum(
    Path(p).stat().st_size
    for p in df_ready["path"]
)

free_bytes = shutil.disk_usage(
    OUTPUT_DATASET.parent
).free

print(
    "Dung lượng cần:",
    round(required_bytes / 1024**3, 2),
    "GB"
)

print(
    "Dung lượng còn trống:",
    round(free_bytes / 1024**3, 2),
    "GB"
)

if free_bytes < required_bytes * 1.15:
    raise RuntimeError("Không đủ dung lượng ổ đĩa")

export_records = []
failed_images = []

# Sử dụng dictionaries thay cho itertuples()
for row in tqdm(
    df_ready.to_dict("records"),
    total=len(df_ready),
    desc="Export EfficientNet"
):
    source = Path(row["path"])

    target_dir = (
        OUTPUT_DATASET
        / row["split"]
        / row["class"]
    )

    target_dir.mkdir(parents=True, exist_ok=True)

    # Tạo tên duy nhất
    relative = str(source.relative_to(SOURCE_DATASET))

    unique_id = hashlib.sha1(
        relative.encode("utf-8")
    ).hexdigest()[:16]

    target = (
        target_dir
        / f"{unique_id}{source.suffix.lower()}"
    )

    try:
        # Kiểm tra ảnh trước khi sao chép
        with Image.open(source) as img:
            img.verify()

        shutil.copy2(source, target)

        export_records.append({
            "source_path": str(source),
            "target_path": str(target),
            "split": row["split"],
            "class": row["class"],
            "original_class": row["original_class"],
            "group_key": row["group_key"]
        })

    except Exception as error:
        failed_images.append({
            "source_path": str(source),
            "error": str(error)
        })

# Xuất báo cáo
export_df = pd.DataFrame(export_records)

export_df.to_csv(
    REPORT_DIR / "efficientnet_export_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    failed_images,
    columns=["source_path", "error"]
).to_csv(
    REPORT_DIR / "efficientnet_failed_images.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nHOÀN THÀNH XUẤT DATASET")
print("Ảnh sao chép thành công:", len(export_records))
print("Ảnh lỗi:", len(failed_images))
print("Đường dẫn:", OUTPUT_DATASET)

In [ ]:
summary = (
    export_df.groupby(["class", "split"])
    .size()
    .unstack(fill_value=0)
)

display(summary)

print("Tổng ảnh đã xuất:", len(export_df))
print("Tổng ảnh bị lỗi:", len(failed_images))

# Kiểm tra nhóm ảnh không xuất hiện ở nhiều split
group_check = (
    export_df.groupby("group_key")["split"]
    .nunique()
)

print(
    "Số nhóm trùng giữa các split:",
    (group_check > 1).sum()
)

In [ ]:
%pip install ImageHash scipy

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import imagehash
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
DATASET_DIR = PROJECT_DIR / "datasets" / "efficientnet"
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

extensions = {".jpg", ".jpeg", ".png", ".webp"}
records = []
errors = []

for split in ["train", "val", "test"]:
    split_dir = DATASET_DIR / split

    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        paths = [
            p for p in class_dir.iterdir()
            if p.is_file() and p.suffix.lower() in extensions
        ]

        for path in tqdm(paths, desc=f"{split}/{class_dir.name}"):
            try:
                with Image.open(path) as img:
                    phash = imagehash.phash(img.convert("RGB"))

                records.append({
                    "split": split,
                    "class": class_dir.name,
                    "path": str(path),
                    "phash": str(phash)
                })

            except Exception as e:
                errors.append({
                    "path": str(path),
                    "error": str(e)
                })

df_hash = pd.DataFrame(records)

df_hash.to_csv(
    REPORT_DIR / "efficientnet_phash.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tổng ảnh kiểm tra:", len(df_hash))
print("Ảnh đọc lỗi:", len(errors))
print("Báo cáo:", REPORT_DIR / "efficientnet_phash.csv")

In [ ]:
hash_groups = (
    df_hash.groupby("phash")
    .agg(
        count=("path", "count"),
        splits=("split", lambda x: sorted(set(x))),
        classes=("class", lambda x: sorted(set(x)))
    )
    .reset_index()
)

suspicious = hash_groups[
    hash_groups["splits"].apply(lambda x: len(x) > 1)
]

print("Số nhóm có pHash giống nhau giữa các split:", len(suspicious))

display(suspicious.head(20))

suspicious.to_csv(
    REPORT_DIR / "phash_cross_split.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

def hash_to_bits(hash_string):
    value = int(hash_string, 16)
    return [(value >> i) & 1 for i in range(64)]

bits = np.array(
    [hash_to_bits(h) for h in df_hash["phash"]],
    dtype=np.float32
)

split_values = df_hash["split"].to_numpy()

train_idx = np.where(split_values == "train")[0]
val_idx = np.where(split_values == "val")[0]
test_idx = np.where(split_values == "test")[0]

# Hamming distance <= 5 tương ứng khoảng cách Euclidean <= sqrt(5)
radius = np.sqrt(5) + 1e-6

pairs = set()

def check_pair_splits(left_idx, right_idx):
    tree = cKDTree(bits[right_idx])

    for source_index in tqdm(left_idx, desc="Kiểm tra gần trùng"):
        neighbors = tree.query_ball_point(
            bits[source_index],
            r=radius
        )

        for local_index in neighbors:
            target_index = right_idx[local_index]

            # Khoảng cách Hamming thực
            distance = int(
                np.count_nonzero(bits[source_index] != bits[target_index])
            )

            if distance <= 5:
                a, b = sorted([int(source_index), int(target_index)])
                pairs.add((a, b, distance))

check_pair_splits(train_idx, val_idx)
check_pair_splits(train_idx, test_idx)
check_pair_splits(val_idx, test_idx)

rows = []

for a, b, distance in sorted(pairs):
    rows.append({
        "image_a": df_hash.iloc[a]["path"],
        "split_a": df_hash.iloc[a]["split"],
        "class_a": df_hash.iloc[a]["class"],
        "image_b": df_hash.iloc[b]["path"],
        "split_b": df_hash.iloc[b]["split"],
        "class_b": df_hash.iloc[b]["class"],
        "hamming_distance": distance
    })

df_near = pd.DataFrame(rows, columns=[
    "image_a", "split_a", "class_a",
    "image_b", "split_b", "class_b",
    "hamming_distance"
])

display(df_near.head(20))

print("Số cặp ảnh nghi gần trùng giữa các split:", len(df_near))

df_near.to_csv(
    REPORT_DIR / "near_duplicate_pairs.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from IPython.display import display
from PIL import Image

for _, row in df_near.head(10).iterrows():
    print("=" * 50)
    print("Khoảng cách:", row["hamming_distance"])
    print("Ảnh A:", row["class_a"], row["split_a"])
    print("Ảnh B:", row["class_b"], row["split_b"])

    for key in ["image_a", "image_b"]:
        with Image.open(row[key]) as img:
            preview = img.convert("RGB")
            preview.thumbnail((250, 250))
            display(preview)

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(
    root=str(DATASET_DIR / "train"),
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root=str(DATASET_DIR / "val"),
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=str(DATASET_DIR / "test"),
    transform=transform
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nDanh sách nhãn:")
print(train_dataset.class_to_idx)

assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert train_dataset.class_to_idx == test_dataset.class_to_idx

loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

images, labels = next(iter(loader))

print("\nBatch shape:", images.shape)
print("Labels:", labels.tolist())
print("CUDA khả dụng:", torch.cuda.is_available())

In [ ]:
import sys
import platform

print("Python:", sys.version)
print("Đường dẫn Python:", sys.executable)
print("Hệ điều hành:", platform.system())
print("Kiến trúc:", platform.architecture()[0])

In [ ]:
%pip install torch torchvision

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Đang sử dụng CPU")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
DATASET_DIR = PROJECT_DIR / "datasets" / "efficientnet"

print("Project tồn tại:", PROJECT_DIR.exists())
print("Dataset tồn tại:", DATASET_DIR.exists())

for split in ["train", "val", "test"]:
    folder = DATASET_DIR / split
    print(f"{split}: {folder.exists()} — {folder}")

In [ ]:
for folder in PROJECT_DIR.rglob("efficientnet"):
    if folder.is_dir():
        print("Tìm thấy:", folder)

In [ ]:
extensions = {".jpg", ".jpeg", ".png", ".webp"}

for split in ["train", "val", "test"]:
    folder = DATASET_DIR / split

    print(f"\n===== {split.upper()} =====")

    if not folder.exists():
        print("Không tìm thấy thư mục")
        continue

    for class_dir in sorted(folder.iterdir()):
        if class_dir.is_dir():
            count = sum(
                p.is_file() and p.suffix.lower() in extensions
                for p in class_dir.iterdir()
            )
            print(class_dir.name, ":", count, "ảnh")

In [ ]:
from pathlib import Path
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
DATASET_DIR = PROJECT_DIR / "datasets" / "efficientnet"

# Sáu lớp hiện có dữ liệu
VALID_CLASSES = [
    "apple__normal_appearance",
    "apple__visible_defect",
    "banana__normal_appearance",
    "banana__visible_defect",
    "orange__normal_appearance",
    "orange__visible_defect",
]

class FoodStateDataset(datasets.ImageFolder):
    def find_classes(self, directory):
        classes = sorted(VALID_CLASSES)
        class_to_idx = {
            name: index
            for index, name in enumerate(classes)
        }
        return classes, class_to_idx

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = FoodStateDataset(
    root=str(DATASET_DIR / "train"),
    transform=transform
)

val_dataset = FoodStateDataset(
    root=str(DATASET_DIR / "val"),
    transform=transform
)

test_dataset = FoodStateDataset(
    root=str(DATASET_DIR / "test"),
    transform=transform
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nClass mapping:")
print(train_dataset.class_to_idx)

assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert train_dataset.class_to_idx == test_dataset.class_to_idx

loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

images, labels = next(iter(loader))

print("\nBatch shape:", images.shape)
print("Labels:", labels.tolist())
print("CUDA khả dụng:", torch.cuda.is_available())

In [ ]:
import torch.nn as nn
from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Đóng băng phần trích xuất đặc trưng
for param in model.features.parameters():
    param.requires_grad = False

# Thay classifier thành 6 lớp
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 6)

model = model.to(device)

print("Device:", device)
print("Classifier:", model.classifier)

In [ ]:
preprocess = weights.transforms()

train_dataset = FoodStateDataset(
    DATASET_DIR / "train",
    transform=preprocess
)

val_dataset = FoodStateDataset(
    DATASET_DIR / "val",
    transform=preprocess
)

test_dataset = FoodStateDataset(
    DATASET_DIR / "test",
    transform=preprocess
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Số lớp:", len(train_dataset.classes))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer: chỉ huấn luyện classifier
optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# Số epochs thử nghiệm
NUM_EPOCHS = 5

print("Device:", device)
print("Epochs:", NUM_EPOCHS)
print("Optimizer:", optimizer.__class__.__name__)
print("Loss:", criterion.__class__.__name__)
print("Số lớp:", len(train_dataset.classes))

In [ ]:
import time
import copy
import pandas as pd
from tqdm.auto import tqdm

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_loss = float("inf")
best_model_state = None

for epoch in range(NUM_EPOCHS):

    start_time = time.time()

    # ===== TRAIN =====
    model.train()

    # Giữ BatchNorm của backbone ở chế độ eval
    model.features.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Train"
    ):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # ===== VALIDATION =====
    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.inference_mode():
        for images, labels in tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Val"
        ):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(dim=1)
            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    # ===== SAVE HISTORY =====
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    # ===== SAVE BEST MODEL =====
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": best_model_state,
            "class_to_idx": train_dataset.class_to_idx,
            "best_val_loss": best_val_loss,
            "architecture": "efficientnet_b0"
        }, MODEL_DIR / "best_efficientnet.pth")

        print("Đã lưu mô hình tốt nhất!")

    elapsed = time.time() - start_time

    print(
        f"\nEpoch {epoch+1}/{NUM_EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Train Acc: {train_acc:.4f}"
        f" | Val Loss: {val_loss:.4f}"
        f" | Val Acc: {val_acc:.4f}"
        f" | Time: {elapsed:.1f}s\n"
    )

# Khôi phục model tốt nhất
if best_model_state is not None:
    model.load_state_dict(best_model_state)

# Lưu lịch sử huấn luyện
history_df = pd.DataFrame(history)
history_df.index = range(1, len(history_df) + 1)
history_df.index.name = "epoch"

history_df.to_csv(
    MODEL_DIR / "training_history.csv",
    encoding="utf-8-sig"
)

print("\nHOÀN THÀNH HUẤN LUYỆN")
print("Best Validation Loss:", best_val_loss)
print("Model:", MODEL_DIR / "best_efficientnet.pth")

In [ ]:
checkpoint_path = MODEL_DIR / "best_efficientnet.pth"

print("File tồn tại:", checkpoint_path.exists())

if checkpoint_path.exists():
    print(
        "Dung lượng:",
        round(checkpoint_path.stat().st_size / 1024**2, 2),
        "MB"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    print("Best Epoch:", checkpoint["epoch"])
    print("Best Val Loss:", checkpoint["best_val_loss"])
    print("Class mapping:", checkpoint["class_to_idx"])

In [ ]:
import matplotlib.pyplot as plt

epochs = history_df.index.to_list()

plt.figure(figsize=(8, 5))
plt.plot(
    epochs,
    history_df["train_loss"],
    label="Training Loss",
    marker="o"
)
plt.plot(
    epochs,
    history_df["val_loss"],
    label="Validation Loss",
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("EfficientNet-B0 - Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    epochs,
    history_df["train_acc"],
    label="Training Accuracy",
    marker="o"
)
plt.plot(
    epochs,
    history_df["val_acc"],
    label="Validation Accuracy",
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("EfficientNet-B0 - Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
%pip install scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

model.eval()

y_true = []
y_pred = []

with torch.inference_mode():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(predictions.cpu().tolist())

print(
    "Validation Accuracy:",
    accuracy_score(y_true, y_pred)
)

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(len(val_dataset.classes))),
        target_names=val_dataset.classes,
        zero_division=0
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=list(range(len(val_dataset.classes))),
    display_labels=val_dataset.classes,
    xticks_rotation=90,
    cmap="Blues"
)

plt.title("EfficientNet-B0 - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

history_path = MODEL_DIR / "training_history.csv"
checkpoint_path = MODEL_DIR / "best_efficientnet.pth"

print("Training history:", history_path.exists())
print("Model checkpoint:", checkpoint_path.exists())

history = pd.read_csv(history_path)

display(history)

print("\nEpoch có Validation Loss thấp nhất:")
best = history.loc[history["val_loss"].idxmin()]
display(best)

In [ ]:
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

model.eval()

y_true = []
y_pred = []

with torch.inference_mode():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(predictions.cpu().tolist())

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation Macro-F1: {macro_f1:.4f}")

print(classification_report(
    y_true,
    y_pred,
    target_names=val_dataset.classes,
    zero_division=0
))

In [ ]:
import pandas as pd

near_path = REPORT_DIR / "near_duplicate_pairs.csv"
exact_path = REPORT_DIR / "phash_cross_split.csv"

for path in [near_path, exact_path]:
    print("\nBáo cáo:", path.name)

    if path.exists():
        df = pd.read_csv(path)
        print("Số dòng nghi vấn:", len(df))
        display(df.head(10))
    else:
        print("Chưa tìm thấy báo cáo.")

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
MODEL_PATH = PROJECT_DIR / "models" / "best_efficientnet.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=True
)

class_to_idx = checkpoint["class_to_idx"]
idx_to_class = {v: k for k, v in class_to_idx.items()}

model = efficientnet_b0(weights=None)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(
    in_features, len(class_to_idx)
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

preprocess = EfficientNet_B0_Weights.DEFAULT.transforms()

print("Đã tải model thành công!")
print("Classes:", idx_to_class)
print("Device:", device)

In [ ]:
from PIL import Image

def predict_food_state(image_path):
    image_path = Path(image_path)

    with Image.open(image_path) as img:
        image = img.convert("RGB")

    tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        outputs = model(tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    confidence, predicted_idx = torch.max(probabilities, dim=0)

    label = idx_to_class[predicted_idx.item()]

    return {
        "label": label,
        "confidence": confidence.item(),
        "probabilities": {
            idx_to_class[i]: round(p.item(), 4)
            for i, p in enumerate(probabilities)
        }
    }

In [ ]:
from IPython.display import display
from PIL import Image

test_image = (
    PROJECT_DIR / "images" / "real_test" / "apple_01.jpg"
)

if not test_image.exists():
    print("Chưa tìm thấy ảnh:", test_image)
else:
    display(Image.open(test_image))

    result = predict_food_state(test_image)

    print("Nhãn AI dự đoán:", result["label"])
    print(f"Confidence: {result['confidence']:.2%}")

    print("\nXác suất của 6 lớp:")
    for label, probability in result["probabilities"].items():
        print(f"{label}: {probability:.2%}")